# AIoT-DA L6: 50 Startups 多元線性回歸分析 (Multiple Linear Regression)
### 採用 CRISP-DM 跨產業資料探勘標準流程
---
本專案嚴格遵循 **CRISP-DM (Cross-Industry Standard Process for Data Mining)** 六大標準階段：
1. **Business Understanding (商業理解)**
2. **Data Understanding (資料理解)**
3. **Data Preparation (資料準備)**
4. **Modeling (模型建立)**
5. **Evaluation (模型評估與前向特徵選取)**
6. **Deployment (模型部署與封裝)**


## 0. 匯入必要套件


In [ ]:
import os
import gzip
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False


## Phase 1: Business Understanding (商業理解)
- **商業目標**：分析並預測 50 家新創公司的獲利（Profit），找出各項支出（R&D, Administration, Marketing）與地理區域（State）對利潤的關鍵影響力。
- **利益關係人**：創業投資人、管理團隊、商業策略顧問。
- **成功標準**：高決定係數（$R^2 \ge 0.90$）以及低均方根誤差（RMSE）與平均絕對誤差（MAE）。


## Phase 2: Data Understanding (資料理解)
載入資料並檢視資料維度、統計摘要、空值分佈。


In [ ]:
data_file = "50_Startups.csv" if os.path.exists("50_Startups.csv") else "50_Startups (2).csv"
df = pd.read_csv(data_file)
print(f"資料形狀: {df.shape}")
display(df.head())
print("\n--- 欄位資訊 ---")
print(df.info())
print("\n--- 缺失值統計 ---")
print(df.isnull().sum())
print("\n--- 敘述統計 ---")
display(df.describe())


## Phase 3: Data Preparation (資料準備)
1. **分離特徵與目標**：$X$ 為前 4 個變數，$y$ 為 Profit。
2. **獨熱編碼 (One-Hot Encoding)**：針對類別欄位 `State`。
   - 設定 `drop_first=True` 避免 **Dummy Variable Trap (虛擬變數陷阱)**。
3. **資料切分 (Train/Test Split)**：80% 訓練集，20% 測試集，`random_state=0`。


In [ ]:
# 1. 分離特徵與目標
X = df.drop("Profit", axis=1)
y = df["Profit"]

# 2. 獨熱編碼 (避開共線性陷阱)
X_encoded = pd.get_dummies(X, columns=["State"], drop_first=True, dtype=int)
print("編碼後特徵欄位:", list(X_encoded.columns))
display(X_encoded.head())

# 3. 切分資料集
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=0)
print(f"訓練集筆數: {len(X_train)}, 測試集筆數: {len(X_test)}")


## Phase 4: Modeling (模型建立)
利用 scikit-learn 的 `LinearRegression` 訓練多元線性回歸模型：
$$\hat{y} = b_0 + b_1 X_1 + b_2 X_2 + \dots + b_k X_k$$


In [ ]:
regressor = LinearRegression()
regressor.fit(X_train, y_train)

print(f"截距項 (b0): {regressor.intercept_:.2f}")
coef_table = pd.DataFrame({
    "特徵名稱": X_encoded.columns,
    "回歸係數 (Weight)": regressor.coef_
})
display(coef_table)


## Phase 5: Evaluation (模型評估與特徵篩選)
計算評估指標 $R^2$, MAE, MSE, RMSE，繪製預測散布圖，並執行前向特徵選取（Forward Feature Selection）。


In [ ]:
y_train_pred = regressor.predict(X_train)
y_test_pred = regressor.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)

print(f"訓練集 R2: {r2_train:.4f}")
print(f"測試集 R2: {r2_test:.6f}")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")

# 繪製 Actual vs Predicted Profit
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, color="#2563EB", edgecolors="k", s=80, alpha=0.8, label="Test Data")
min_val, max_val = min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label="Perfect Fit (y = x)")
plt.xlabel("Actual Profit (實際利潤)", fontsize=12)
plt.ylabel("Predicted Profit (預測利潤)", fontsize=12)
plt.title("Actual vs Predicted Profit", fontsize=14, fontweight="bold")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()


### Forward Feature Selection (前向逐步特徵選取實驗)
由 1 個特徵遞增至 5 個特徵，驗證哪些特徵組具有最佳的泛化能力：


In [ ]:
def evaluate_performance(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred)), r2_score(y_true, y_pred)

def forward_selection(X_tr, X_te, y_tr, y_te, max_features):
    selected = []
    results = []
    all_cols = list(X_tr.columns)
    for i in range(1, max_features + 1):
        best_rmse, best_r2, best_feat = np.inf, -np.inf, None
        for feat in [c for c in all_cols if c not in selected]:
            curr_cols = selected + [feat]
            lr = LinearRegression().fit(X_tr[curr_cols], y_tr)
            c_rmse, c_r2 = evaluate_performance(y_te, lr.predict(X_te[curr_cols]))
            if c_rmse < best_rmse:
                best_rmse, best_r2, best_feat = c_rmse, c_r2, feat
        selected.append(best_feat)
        results.append({
            "Number of Features": i,
            "Selected Features": selected.copy(),
            "RMSE": round(best_rmse, 6),
            "R-squared": round(best_r2, 6)
        })
    return pd.DataFrame(results)

results_df = forward_selection(X_train, X_test, y_train, y_test, max_features=5)
print(tabulate(results_df, headers="keys", tablefmt="pipe"))


## Phase 6: Deployment (模型部署與封裝)
使用 `gzip` 配合 `pickle` 壓縮持久化存檔，並模擬部署後載入進行新創預測。


In [ ]:
# 匯出封裝
model_file = "startup_lr_model.pkl.gz"
with gzip.open(model_file, "wb") as f:
    pickle.dump({"model": regressor, "features": list(X_encoded.columns)}, f)
print(f"模型成功匯出至: {model_file}")

# 載入並進行預測
with gzip.open(model_file, "rb") as f:
    data_pack = pickle.load(f)
    model_loaded = data_pack["model"]
    features_loaded = data_pack["features"]

sample = pd.DataFrame([{
    "R&D Spend": 160000.0,
    "Administration": 130000.0,
    "Marketing Spend": 300000.0,
    "State_Florida": 1,
    "State_New York": 0
}], columns=features_loaded)

pred = model_loaded.predict(sample)[0]
print(f"新創公司預測利潤: ${pred:,.2f}")
